DDL setup for gold dimension tables in `adwm_wh.gold`.

Notebook contents:

* Table definitions for `DimCustomer`, `DimProduct`, `DimEmployee`, `DimDate`, and `DimTime`
* Identity-based surrogate keys for customer, product, and employee dimensions
* Static population logic for the calendar and time dimensions

Design notes:

* Business-key uniqueness for customer, product, and employee dimensions is enforced by load logic in the corresponding dimension notebooks
* `DimDate` is populated for dates from `2000-01-01` through `2035-12-31`
* `DimTime` is populated at one-second grain for a full 24-hour day

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adwm_wh.gold;

CREATE OR REPLACE TABLE adwm_wh.gold.DimCustomer (
    CustomerKey       BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID        INT NOT NULL,
    CustomerType      STRING NOT NULL,
    FullName          STRING,
    AccountNumber     STRING,
    TerritoryName     STRING,
    CountryRegion     STRING,
    modified_date     TIMESTAMP NOT NULL DEFAULT current_timestamp(),
    CONSTRAINT dimcustomer_pk PRIMARY KEY (CustomerKey)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
CREATE OR REPLACE TABLE adwm_wh.gold.DimProduct (
    ProductKey        BIGINT GENERATED ALWAYS AS IDENTITY,
    ProductID         INT NOT NULL,
    ProductNumber     STRING NOT NULL,
    ProductName       STRING NOT NULL,
    Color             STRING,
    Size              STRING,
    StandardCost      DECIMAL(19,4),
    ListPrice         DECIMAL(19,4),
    SubcategoryName   STRING,
    CategoryName      STRING,
    modified_date     TIMESTAMP NOT NULL DEFAULT current_timestamp(),
    CONSTRAINT dimproduct_pk PRIMARY KEY (ProductKey)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
CREATE OR REPLACE TABLE adwm_wh.gold.DimEmployee (
    EmployeeKey       BIGINT GENERATED ALWAYS AS IDENTITY,
    EmployeeID        INT NOT NULL,
    FullName          STRING NOT NULL,
    JobTitle          STRING,
    Gender            STRING,
    HireDate          DATE,
    DepartmentName    STRING,
    DepartmentGroup   STRING,
    IsActive          BOOLEAN NOT NULL,
    modified_date     TIMESTAMP NOT NULL DEFAULT current_timestamp(),
    CONSTRAINT dimemployee_pk PRIMARY KEY (EmployeeKey)
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
CREATE OR REPLACE TABLE adwm_wh.gold.DimDate (
    DateKey             INT NOT NULL,
    FullDate            DATE NOT NULL,
    DayOfWeekNumber     INT NOT NULL,
    DayName             STRING NOT NULL,
    DayOfMonthNumber    INT NOT NULL,
    DayOfYearNumber     INT NOT NULL,
    WeekOfYearNumber    INT NOT NULL,
    MonthNumber         INT NOT NULL,
    MonthName           STRING NOT NULL,
    QuarterNumber       INT NOT NULL,
    YearNumber          INT NOT NULL,
    IsWeekend           BOOLEAN NOT NULL,
    CONSTRAINT dimdate_pk PRIMARY KEY (DateKey)
)
USING DELTA;

INSERT OVERWRITE adwm_wh.gold.DimDate (
    DateKey,
    FullDate,
    DayOfWeekNumber,
    DayName,
    DayOfMonthNumber,
    DayOfYearNumber,
    WeekOfYearNumber,
    MonthNumber,
    MonthName,
    QuarterNumber,
    YearNumber,
    IsWeekend
)
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS DateKey,
    d AS FullDate,
    dayofweek(d) AS DayOfWeekNumber,
    date_format(d, 'EEEE') AS DayName,
    dayofmonth(d) AS DayOfMonthNumber,
    dayofyear(d) AS DayOfYearNumber,
    weekofyear(d) AS WeekOfYearNumber,
    month(d) AS MonthNumber,
    date_format(d, 'MMMM') AS MonthName,
    quarter(d) AS QuarterNumber,
    year(d) AS YearNumber,
    CASE WHEN dayofweek(d) IN (1, 7) THEN TRUE ELSE FALSE END AS IsWeekend
FROM (
    SELECT explode(sequence(to_date('2000-01-01'), to_date('2035-12-31'), interval 1 day)) AS d
);

In [0]:
%sql
CREATE OR REPLACE TABLE adwm_wh.gold.DimTime (
    TimeKey             INT NOT NULL,
    FullTime            STRING NOT NULL,
    Hour24              INT NOT NULL,
    MinuteNumber        INT NOT NULL,
    SecondNumber        INT NOT NULL,
    Hour12              INT NOT NULL,
    AmPm                STRING NOT NULL,
    CONSTRAINT dimtime_pk PRIMARY KEY (TimeKey)
)
USING DELTA;

INSERT OVERWRITE adwm_wh.gold.DimTime (
    TimeKey,
    FullTime,
    Hour24,
    MinuteNumber,
    SecondNumber,
    Hour12,
    AmPm
)
WITH seconds AS (
    SELECT id AS second_of_day
    FROM range(0, 86400)
),
parts AS (
    SELECT
        CAST(floor(second_of_day / 3600) AS INT) AS hour24,
        CAST(floor((second_of_day % 3600) / 60) AS INT) AS minute_number,
        CAST(second_of_day % 60 AS INT) AS second_number
    FROM seconds
)
SELECT
    CAST(
        concat(
            lpad(CAST(hour24 AS STRING), 2, '0'),
            lpad(CAST(minute_number AS STRING), 2, '0'),
            lpad(CAST(second_number AS STRING), 2, '0')
        ) AS INT
    ) AS TimeKey,
    concat(
        lpad(CAST(hour24 AS STRING), 2, '0'), ':',
        lpad(CAST(minute_number AS STRING), 2, '0'), ':',
        lpad(CAST(second_number AS STRING), 2, '0')
    ) AS FullTime,
    hour24 AS Hour24,
    minute_number AS MinuteNumber,
    second_number AS SecondNumber,
    CASE WHEN pmod(hour24, 12) = 0 THEN 12 ELSE pmod(hour24, 12) END AS Hour12,
    CASE WHEN hour24 < 12 THEN 'AM' ELSE 'PM' END AS AmPm
FROM parts;